# 2026 COMP90042 Project
*Make sure you change the file name with your group id.*

# Readme
*If there is something to be noted for the marker, please mention here.*

*If you are planning to implement a program with Object Oriented Programming style, please put those the bottom of this ipynb file*

# 1.DataSet Processing


In [268]:
# Install required packages, including rank_bm25 for BM25-based retrieval.
!pip install rank_bm25 transformers accelerate -q

In [269]:
# Import necessary packages and modules.
import os
import json
import re
import random
import numpy as np
from tqdm import tqdm

import torch
from torch.utils.data import Dataset

from rank_bm25 import BM25Okapi
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

## 1.1 Loading Climate Claim and Evidence Datasets
在本项目中，我们首先加载 climate claim verification 任务所需的所有数据文件，包括：

* training set
* development set
* test set
* baseline predictions
* evidence corpus

其中：

* claim datasets 保存了 claim text、claim label 以及对应的 evidence ids
* evidence.json 保存了所有 evidence passages 的具体文本内容

为了方便后续 retrieval 与 classification，我们首先将所有 JSON 文件统一读取为 Python dictionary 格式，并对数据集规模进行基础统计，同时随机展示部分 evidence examples 以验证数据读取是否正确。

随后，我们进一步对 claim labels 进行 numerical encoding，将字符串标签转换为模型可处理的整数形式：

* SUPPORTS → 0
* REFUTES → 1
* NOT_ENOUGH_INFO → 2
* DISPUTED → 3

最后，我们将原始 claim data 转换为结构化的 claim-level examples。对于 train 和 dev datasets，系统直接使用数据集中提供的 gold evidence ids；而对于 test dataset，由于不存在 ground-truth evidence，系统会自动调用 BM25 retrieval 获取 top-k candidate evidences。

每个 example 会统一保存：

* claim text
* evidence ids
* evidence texts
* concatenated evidence sequence

其中多个 evidence passages 会通过 [SEP] token 拼接，以构建 transformer-based classifier 所需的 claim-evidence representation。

这一 preprocessing step 的主要目标是：

* 统一 train/dev/test 数据格式
* 保持 claim 与 evidence 的对应关系
* 构建模型可直接输入的 structured examples
* 为后续 micro-level classification 提供标准化输入

In [270]:
# =========================
# File Paths
# =========================
DATA_DIR = "/content"

train_path = os.path.join(DATA_DIR, "train-claims.json")
dev_path = os.path.join(DATA_DIR, "dev-claims.json")
test_path = os.path.join(DATA_DIR, "test-claims-unlabelled.json")
baseline_path = os.path.join(DATA_DIR, "dev-claims-baseline.json")
evidence_path = os.path.join(DATA_DIR, "evidence.json")


# =========================
# Generic JSON Loader
# =========================
def load_json(path):
    """
    Load any JSON file from disk.

    Args:
        path (str): file path

    Returns:
        dict: loaded JSON content
    """
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


# =========================
# Load All Project Data
# =========================
print("Loading datasets...")

train_claims = load_json(train_path)
dev_claims = load_json(dev_path)
test_claims = load_json(test_path)
baseline_claims = load_json(baseline_path)
evidence_dict = load_json(evidence_path)


# =========================
# Basic Statistics
# =========================
print("Train claims:", len(train_claims))
print("Dev claims:", len(dev_claims))
print("Test claims:", len(test_claims))
print("Baseline predictions:", len(baseline_claims))
print("Evidence passages:", len(evidence_dict))


# =========================
# Preview Evidence Samples
# =========================
print("\nSample evidence examples:")

for i, (eid, text) in enumerate(evidence_dict.items()):
    print(f"{eid}: {text[:200]}")
    if i >= 2:
        break

Loading datasets...
Train claims: 1228
Dev claims: 154
Test claims: 153
Baseline predictions: 154
Evidence passages: 1208827

Sample evidence examples:
evidence-0: John Bennet Lawes, English entrepreneur and agricultural scientist
evidence-1: Lindberg began his professional career at the age of 16, eventually moving to New York City in 1977.
evidence-2: ``Boston (Ladies of Cambridge)'' by Vampire Weekend


In [272]:
# Mapping from label names to integer IDs
# This is used to convert string labels into numerical format for model training
label2id = {
    "SUPPORTS": 0,
    "REFUTES": 1,
    "NOT_ENOUGH_INFO": 2,
    "DISPUTED": 3
}

# Reverse mapping from IDs back to label names
# This is useful for converting model predictions (integers) back to human-readable labels
id2label = {v: k for k, v in label2id.items()}

In [273]:
def build_examples(claims_dict, labelled=True):
    """
    Build claim-level examples.

    For train/dev:
        Use provided gold evidence IDs.

    For test:
        No labels and often no gold evidence,
        so fallback to BM25 retrieval.
    """
    examples = []

    for claim_id, item in tqdm(claims_dict.items()):
        claim_text = item["claim_text"]

        # ==========================================
        # If gold evidences exist (train/dev)
        # ==========================================
        if "evidences" in item:
            evidence_ids_for_claim = item["evidences"][:5]

            evidence_texts_for_claim = [
                evidence_dict[eid] for eid in evidence_ids_for_claim
            ]

        # ==========================================
        # If no evidences exist (test set)
        # ==========================================
        else:
            evidence_ids_for_claim, evidence_texts_for_claim = retrieve_top_k_evidence(
                claim_text,
                top_k=5
            )

        example = {
            "claim_id": claim_id,
            "claim_text": claim_text,
            "evidence_ids": evidence_ids_for_claim,
            "evidence_texts": evidence_texts_for_claim,
            "evidence_text": " [SEP] ".join(evidence_texts_for_claim)
        }

        if labelled and "claim_label" in item:
            example["label"] = label2id[item["claim_label"]]

        examples.append(example)

    return examples

## 1.2 BM25 Evidence Retrieval
由于 climate claim verification 任务需要从大量 evidence passages 中寻找与 claim 最相关的证据，我们首先构建了一个基于 BM25 的 lexical retrieval system。

具体而言：

* 首先对所有 evidence 文本进行 tokenization
* 将文本统一转为 lowercase
* 仅保留字母与数字 token
* 去除标点与特殊符号

随后，我们使用 BM25Okapi 对整个 evidence corpus 建立检索索引。

该 retrieval module 的作用是：在 test set 没有 gold evidence 时 自动为 claim 检索 top-k relevant evidence passages 为后续 BERT classifier 提供 candidate evidence inputs

In [271]:
import re
import numpy as np
from tqdm import tqdm
from rank_bm25 import BM25Okapi

def simple_tokenize(text):
    """
    Tokenizer for BM25.
    Lowercase text and keep only words/numbers.
    """
    text = text.lower()
    tokens = re.findall(r"[a-z0-9]+", text)
    return tokens

# Acquire evidence id and text
evidence_ids = list(evidence_dict.keys())
evidence_texts = list(evidence_dict.values())

print("Tokenizing evidence corpus...")

# Tokenize each evidence document
tokenized_evidence = [
    simple_tokenize(text)
    for text in tqdm(evidence_texts)
]

print("Building BM25 index...")

# Build BM25 model over the tokenized corpus
# This object will be used to compute relevance scores between a query and documents
bm25 = BM25Okapi(tokenized_evidence)

print("BM25 index built.")

Tokenizing evidence corpus...


100%|██████████| 1208827/1208827 [00:50<00:00, 23923.29it/s]


Building BM25 index...
BM25 index built.


## 1.3 Constructing Model-ready Claim–Evidence Inputs
在完成 preprocessing pipeline 后，我们进一步构建 train、development 与 test datasets 对应的 model-ready claim-evidence examples。

对于 train 与 dev datasets，系统会保留 gold claim labels 并直接使用数据集中提供的 evidence passages；而对于 test dataset，系统仅构建 claim-evidence inputs，不包含 ground-truth labels。

最终生成的 examples 会统一包含：

* claim text
* evidence ids
* evidence texts
* concatenated evidence sequence

这些 structured examples 将作为后续 BERT-based micro classifier 的输入。

In [274]:
train_claims_small = dict(list(train_claims.items()))
dev_claims_small = dict(list(dev_claims.items()))
test_claims_small = dict(list(test_claims.items()))

train_examples = build_examples(
    train_claims_small,
    labelled=True
)

dev_examples = build_examples(
    dev_claims_small,
    labelled=True
)

test_examples = build_examples(
  test_claims_small,
  labelled=False
)

print("Train examples:", len(train_examples))
print("Dev examples:", len(dev_examples))
print("Test examples:", len(test_examples))

print("\nSample processed example:")
print(train_examples[0])

100%|██████████| 154/154 [00:00<00:00, 159329.75it/s]

Train examples: 1228
Dev examples: 154

Sample processed example:
{'claim_id': 'claim-1937', 'claim_text': 'Not only is there no scientific evidence that CO2 is a pollutant, higher CO2 concentrations actually help ecosystems support more plant and animal life.', 'evidence_ids': ['evidence-442946', 'evidence-1194317', 'evidence-12171'], 'evidence_texts': ['At very high concentrations (100 times atmospheric concentration, or greater), carbon dioxide can be toxic to animal life, so raising the concentration to 10,000 ppm (1%) or higher for several hours will eliminate pests such as whiteflies and spider mites in a greenhouse.', 'Plants can grow as much as 50 percent faster in concentrations of 1,000 ppm CO 2 when compared with ambient conditions, though this assumes no change in climate and no limitation on other nutrients.', 'Higher carbon dioxide concentrations will favourably affect plant growth and demand for water.'], 'evidence_text': 'At very high concentrations (100 times atmospher

# 2.Model Implementation
(You can add as many code blocks and text blocks as you need. However, YOU SHOULD NOT MODIFY the section title)




## 2.1 Micro-level Classifer

### 2.1.1 Building Sentence-level Micro Examples

In [ ]:
micro_label2id = {
    "SUPPORTS": 0,
    "REFUTES": 1,
    "NOT_ENOUGH_INFO": 2
}

micro_id2label = {v: k for k, v in micro_label2id.items()}

In [275]:
def build_micro_examples(examples, labelled=True, skip_disputed=True):
    """
    Convert claim-level examples into sentence-level micro examples.

    Each claim with provided evidence becomes multiple micro examples:
        claim + evidence_i -> micro label

    DISPUTED is skipped during micro training because it is a macro-level label.
    """
    micro_examples = []

    for ex in examples:
        claim_id = ex["claim_id"]
        claim_text = ex["claim_text"]

        evidence_ids = ex["evidence_ids"]
        evidence_texts = ex["evidence_texts"]

        if labelled:
            macro_label = id2label[ex["label"]]

            if skip_disputed and macro_label == "DISPUTED":
                continue

            if macro_label not in micro_label2id:
                continue

            micro_label = micro_label2id[macro_label]

        for rank, (eid, ev_text) in enumerate(zip(evidence_ids, evidence_texts)):
            item = {
                "claim_id": claim_id,
                "claim_text": claim_text,
                "evidence_id": eid,
                "evidence_text": ev_text,
                "rank": rank
            }

            if labelled:
                item["label"] = micro_label

            micro_examples.append(item)

    return micro_examples


train_micro_examples = build_micro_examples(
    train_examples,
    labelled=True,
    skip_disputed=True
)

dev_micro_examples = build_micro_examples(
    dev_examples,
    labelled=True,
    skip_disputed=True
)

print("Train micro examples:", len(train_micro_examples))
print("Dev micro examples:", len(dev_micro_examples))
print(train_micro_examples[0])

Train micro examples: 3730
Dev micro examples: 433
{'claim_id': 'claim-126', 'claim_text': 'El Niño drove record highs in global temperatures suggesting rise may not be down to man-made emissions.', 'evidence_id': 'evidence-338219', 'evidence_text': 'While ‘climate change’ can be due to natural forces or human activity, there is now substantial evidence to indicate that human activity – and specifically increased greenhouse gas (GHGs) emissions – is a key factor in the pace and extent of global temperature increases.', 'rank': 0, 'label': 1}


### 2.1.2 BERT-based Micro Classifier
在 micro-level classification 阶段，我们采用 bert-base-uncased 作为基础 transformer encoder，用于完成 sentence-level claim-evidence entailment prediction。对于每一个 micro example，系统会将 claim text + evidence text 作为 sentence pair 输入到 BERT tokenizer 中。随后，tokenizer 会自动生成：

* input_ids
* attention_mask
* token_type_ids

等 transformer 所需的输入 tensors。其中：

* claim_text 被视为 sentence A
* evidence_text 被视为 sentence B

从而使 BERT 能够学习 claim 与 evidence 之间的语义关系。

为了适配 batch training，我们采用：

* padding="max_length"
* truncation=True

并将 maximum sequence length 设置为 256，以统一输入长度并降低 GPU memory consumption。随后，我们进一步构建 MicroClaimEvidenceDataset，将所有 sentence-level micro examples 转换为 PyTorch Dataset 格式，方便后续 HuggingFace Trainer 进行 fine-tuning。

最后，我们使用 AutoModelForSequenceClassification 构建 BERT-based micro classifier，并输出三个 logits，对应：
* SUPPORTS
* REFUTES
* NOT_ENOUGH_INFO

从而完成 sentence-level entailment prediction。

In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

MAX_LENGTH = 256
BATCH_SIZE = 8
LEARNING_RATE = 2e-5
EPOCHS = 1

In [277]:
class MicroClaimEvidenceDataset(Dataset):
    """
    Dataset for sentence-level BERT micro-verdict prediction.

    Input:
        claim_text + one evidence sentence

    Output:
        SUPPORTS / REFUTES / NOT_ENOUGH_INFO
    """

    def __init__(self, micro_examples, tokenizer, max_length=256, labelled=True):
        self.examples = micro_examples
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.labelled = labelled

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        ex = self.examples[idx]

        encoding = self.tokenizer(
            ex["claim_text"],
            ex["evidence_text"],
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )

        item = {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
        }

        if "token_type_ids" in encoding:
            item["token_type_ids"] = encoding["token_type_ids"].squeeze(0)

        if self.labelled:
            item["labels"] = torch.tensor(ex["label"], dtype=torch.long)

        return item
train_micro_dataset = MicroClaimEvidenceDataset(
    train_micro_examples,
    tokenizer,
    max_length=MAX_LENGTH,
    labelled=True
)

dev_micro_dataset = MicroClaimEvidenceDataset(
    dev_micro_examples,
    tokenizer,
    max_length=MAX_LENGTH,
    labelled=True
)

print("Train micro dataset:", len(train_micro_dataset))
print("Dev micro dataset:", len(dev_micro_dataset))

Train micro dataset: 3730
Dev micro dataset: 433


In [278]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    id2label=micro_id2label,
    label2id=micro_label2id
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


### 2.1.3 Training Strategy and Evaluation Metrics
在训练阶段，我们首先考虑 micro-level labels 存在类别不平衡的问题。由于不同 entailment labels 的样本数量差异较大，如果直接使用 standard cross-entropy loss，模型可能会过度偏向 majority class。因此，我们根据训练集中各类别样本数量计算 class weights，并将其加入 weighted cross-entropy loss 中，以增强模型对 minority labels 的学习能力。

随后，我们进一步扩展 HuggingFace Trainer，实现 WeightedLossTrainer。该 trainer 会在 loss computation 阶段自动使用预先计算好的 class weights，从而缓解 label imbalance 对训练稳定性的影响。

在 evaluation 部分，我们使用：

* accuracy
* macro F1
* weighted F1

作为主要评估指标。其中：

* accuracy 用于衡量整体预测正确率
* macro F1 更关注不同类别之间的 balanced performance
* weighted F1 会考虑类别样本数量差异

由于本任务存在类别不平衡问题，因此我们主要使用macro F1 作为 selecting best model 的核心指标。

最后，我们使用 HuggingFace TrainingArguments 配置训练过程，包括：
* learning rate
* batch size
* number of epochs
* checkpoint saving strategy
* evaluation strategy

并通过 Trainer 对 micro-level BERT classifier 进行 fine-tuning。训练完成后，系统会自动在 development set 上进行 evaluation，并输出最终的 micro-level classification performance。

In [276]:
from collections import Counter
import torch
import numpy as np

train_label_counts = Counter([ex["label"] for ex in train_micro_examples])

num_classes = len(micro_label2id)
total_count = sum(train_label_counts.values())

class_weights = []

for i in range(num_classes):
    weight = total_count / (num_classes * train_label_counts[i])
    class_weights.append(weight)

class_weights = torch.tensor(class_weights, dtype=torch.float)

print("Class weights:", class_weights)

Class weights: tensor([0.9258, 2.7206, 0.6442])


In [279]:
from transformers import Trainer
import torch.nn as nn

class WeightedLossTrainer(Trainer):
    def __init__(self, class_weights=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)

        logits = outputs.logits

        loss_fn = nn.CrossEntropyLoss(
            weight=self.class_weights.to(logits.device)
        )

        loss = loss_fn(logits, labels)

        return (loss, outputs) if return_outputs else loss

In [280]:
from sklearn.metrics import accuracy_score, f1_score

def compute_micro_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    accuracy = accuracy_score(labels, preds)
    macro_f1 = f1_score(labels, preds, average="macro")
    weighted_f1 = f1_score(labels, preds, average="weighted")

    return {
        "micro_accuracy": accuracy,
        "micro_macro_f1": macro_f1,
        "micro_weighted_f1": weighted_f1
    }

In [281]:
training_args = TrainingArguments(
    output_dir="./micro_macro_bert",

    eval_strategy="epoch",
    save_strategy="epoch",

    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=EPOCHS,
    weight_decay=0.01,

    logging_dir="./logs",
    logging_steps=50,

    load_best_model_at_end=True,
    metric_for_best_model="micro_macro_f1",#"micro_accuracy",
    greater_is_better=True,

    save_total_limit=2,
    report_to="none"
)
trainer = WeightedLossTrainer(
    model=model,
    args=training_args,
    train_dataset=train_micro_dataset,
    eval_dataset=dev_micro_dataset,
    compute_metrics=compute_micro_metrics,
    class_weights=class_weights
)
trainer.train()
micro_dev_results = trainer.evaluate()
print(micro_dev_results)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Epoch,Training Loss,Validation Loss,Micro Accuracy,Micro Macro F1,Micro Weighted F1
1,0.841940,0.942876,0.577367,0.519927,0.574646


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

{'eval_loss': 0.9428756833076477, 'eval_micro_accuracy': 0.5773672055427251, 'eval_micro_macro_f1': 0.5199271402550091, 'eval_micro_weighted_f1': 0.5746463231489545, 'eval_runtime': 6.5358, 'eval_samples_per_second': 66.251, 'eval_steps_per_second': 8.415, 'epoch': 1.0}


## 2.2 Macro-level Classifer

### 2.2.1 Building Macro-level Features from Micro Predictions

在完成 micro-level classifier 训练后，我们进一步构建 macro-level aggregation features。Macro-level classifier 的目标不是判断单条 evidence 与 claim 的关系，而是综合多个 evidence-level predictions，预测最终的 claim-level label——SUPPORTS / REFUTES / NOT_ENOUGH_INFO / DISPUTED。

首先，我们使用已经训练好的 micro BERT classifier 对每个 claim 对应的多条 evidence 分别进行预测，并提取每条 evidence 的三分类概率分布：P(SUPPORTS), P(REFUTES), P(NOT_ENOUGH_INFO)，因此，每个 claim 会被表示为一个 micro prediction matrix：[num_evidence, 3]，其中每一行对应一条 evidence 的 micro-level prediction probabilities。

为了帮助 macro model 判断 evidence 之间是否存在冲突，我们进一步计算 uncertainty-related features，例如 Jensen-Shannon divergence。该特征用于衡量多条 evidence predictions 之间的分歧程度。如果不同 evidence 同时表现出较强的 SUPPORTS 与 REFUTES signals，则该 claim 更可能属于 DISPUTED 类别。

随后，我们将每个 claim 的 micro probabilities、uncertainty features、evidence ids 和 macro label 封装成 macro-level feature examples。由于不同 claim 的 evidence 数量可能不同，我们在 MacroAggregationDataset 中对 evidence 维度进行 padding 或 truncation，并使用 evidence_mask 区分真实 evidence 与 padding 部分。

最后，这些 features 被转换为 PyTorch Dataset 和 DataLoader，用于后续 macro-level aggregation model 的训练。

In [282]:
macro_label2id = {
    "SUPPORTS": 0,
    "REFUTES": 1,
    "NOT_ENOUGH_INFO": 2,
    "DISPUTED": 3
}

macro_id2label = {v: k for k, v in macro_label2id.items()}

def entropy_np(probs):
    probs = np.array(probs, dtype=np.float32)
    return -np.sum(probs * np.log(probs + 1e-8))


def js_divergence_np(prob_list):
    prob_array = np.array(prob_list, dtype=np.float32)
    mean_prob = prob_array.mean(axis=0)
    return entropy_np(mean_prob) - np.mean([entropy_np(p) for p in prob_array])


def support_refute_variance_np(prob_list):
    prob_array = np.array(prob_list, dtype=np.float32)
    return np.var(prob_array[:, 0]) + np.var(prob_array[:, 1])
def get_micro_probs_for_claim(claim_text, evidence_texts):
    """
    Use trained micro BERT to predict probs for all evidences of one claim.
    Output shape: [num_evidence, 3]
    """
    model.eval()
    all_probs = []

    for ev_text in evidence_texts:
        encoding = tokenizer(
            claim_text,
            ev_text,
            truncation=True,
            padding="max_length",
            max_length=MAX_LENGTH,
            return_tensors="pt"
        )

        encoding = {k: v.to(model.device) for k, v in encoding.items()}

        with torch.no_grad():
            outputs = model(**encoding)
            temperature = 1
            probs = torch.softmax(outputs.logits/temperature, dim=-1).squeeze(0)

        all_probs.append(probs.cpu().numpy())

    return np.array(all_probs, dtype=np.float32)
def build_macro_features(examples, labelled=True):
    """
    Build claim-level features for learned attention aggregation.

    Each claim becomes:
        micro_probs: [num_evidence, 3]
        uncertainty_features: [JSD, support_refute_variance]
        label: 4-class macro label
    """
    feature_examples = []

    for ex in examples:
        claim_text = ex["claim_text"]
        evidence_texts = ex["evidence_texts"]

        micro_probs = get_micro_probs_for_claim(claim_text, evidence_texts)

        jsd = js_divergence_np(micro_probs)
        sr_var = support_refute_variance_np(micro_probs)

        item = {
            "claim_id": ex["claim_id"],
            "evidence_ids": ex["evidence_ids"],
            "micro_probs": micro_probs,
            "uncertainty_features": np.array([jsd], dtype=np.float32)
        }

        if labelled:
            macro_label = id2label[ex["label"]]
            item["label"] = macro_label2id[macro_label]

        feature_examples.append(item)

    return feature_examples
train_macro_features = build_macro_features(train_examples, labelled=True)
dev_macro_features = build_macro_features(dev_examples, labelled=True)

print("Train macro features:", len(train_macro_features))
print("Dev macro features:", len(dev_macro_features))
print(train_macro_features[0].keys())

Train macro features: 1228
Dev macro features: 154
dict_keys(['claim_id', 'evidence_ids', 'micro_probs', 'uncertainty_features', 'label'])


In [283]:
from torch.utils.data import DataLoader
class MacroAggregationDataset(Dataset):
    def __init__(self, feature_examples, labelled=True, max_evidence=5):
        self.examples = feature_examples
        self.labelled = labelled
        self.max_evidence = max_evidence

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        ex = self.examples[idx]

        micro_probs = torch.tensor(ex["micro_probs"], dtype=torch.float32)

        # pad / truncate to fixed number of evidence
        num_evidence = micro_probs.shape[0]

        if num_evidence > self.max_evidence:
            micro_probs = micro_probs[:self.max_evidence]
            evidence_mask = torch.ones(self.max_evidence, dtype=torch.float32)

        else:
            pad_size = self.max_evidence - num_evidence
            pad = torch.zeros(pad_size, micro_probs.shape[1])
            micro_probs = torch.cat([micro_probs, pad], dim=0)

            evidence_mask = torch.cat([
                torch.ones(num_evidence, dtype=torch.float32),
                torch.zeros(pad_size, dtype=torch.float32)
            ])

        item = {
            "claim_id": ex["claim_id"],
            "evidence_ids": ex["evidence_ids"],
            "micro_probs": micro_probs,
            "evidence_mask": evidence_mask,
            "uncertainty_features": torch.tensor(ex["uncertainty_features"], dtype=torch.float32)
        }

        if self.labelled:
            item["labels"] = torch.tensor(ex["label"], dtype=torch.long)

        return item
def macro_collate_fn(batch):
    output = {
        "claim_ids": [x["claim_id"] for x in batch],
        "evidence_ids": [x["evidence_ids"] for x in batch],
        "micro_probs": torch.stack([x["micro_probs"] for x in batch]),
        "evidence_mask": torch.stack([x["evidence_mask"] for x in batch]),
        "uncertainty_features": torch.stack([x["uncertainty_features"] for x in batch]),
    }

    if "labels" in batch[0]:
        output["labels"] = torch.stack([x["labels"] for x in batch])

    return output

train_macro_dataset = MacroAggregationDataset(train_macro_features, labelled=True)
dev_macro_dataset = MacroAggregationDataset(dev_macro_features, labelled=True)

train_macro_loader = DataLoader(
    train_macro_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=macro_collate_fn
)

dev_macro_loader = DataLoader(
    dev_macro_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=macro_collate_fn
)

### 2.2.2 Attention-based Macro Aggregator Training
在 macro-level model 中，我们采用 learned attention aggregation 的方式对多条 evidence 的 micro-level prediction probabilities 进行加权汇总。相比简单平均或 majority voting，attention mechanism 可以让模型自动学习哪些 evidence 对最终 claim-level prediction 更重要。

具体而言，模型首先对每条 evidence 的 micro probability vector 计算 attention score，并通过 softmax 得到 normalized attention weights。随后，模型根据 attention weights 对所有 evidence-level probabilities 进行加权求和，得到一个 claim-level weighted probability representation。

接着，我们将该 weighted probability representation 与 uncertainty features 拼接，作为最终 macro classifier 的输入。该 classifier 输出四个 logits，对应：SUPPORTS / REFUTES / NOT_ENOUGH_INFO / DISPUTED。

其中，DISPUTED 类别需要依赖多个 evidence 之间的综合关系，因此更适合在 macro-level 阶段预测，而不是在 micro-level classifier 中直接预测。

训练阶段，我们使用 AdamW optimizer 对 macro aggregator 进行训练，并在每个 epoch 后在 development set 上计算 macro-level accuracy。通过这一过程，macro model 能够学习如何从多个 sentence-level predictions 中聚合出最终的 claim-level decision。

In [284]:
import torch.nn as nn

macro_hidden_size = 64
macro_dropout = 0.5
macro_lr = 5e-3
macro_weight_decay = 0.01
uncertainty_dim = 1

class LearnedAttentionMacroAggregator(nn.Module):
    def __init__(
        self,
        num_micro_labels=3,
        uncertainty_dim=1,
        hidden_size=32,
        dropout=0.1,
        num_macro_labels=4
    ):
        super().__init__()

        self.attention = nn.Sequential(
            nn.Linear(num_micro_labels, hidden_size),
            nn.Tanh(),
            nn.Linear(hidden_size, 1)
        )

        self.classifier = nn.Sequential(
            nn.Linear(num_micro_labels + uncertainty_dim, hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, num_macro_labels)
        )

    def forward(self, micro_probs, uncertainty_features, evidence_mask=None, labels=None):
        attn_logits = self.attention(micro_probs).squeeze(-1)

        if evidence_mask is not None:
            attn_logits = attn_logits.masked_fill(evidence_mask == 0, -1e9)

        attn_weights = torch.softmax(attn_logits, dim=1)

        weighted_probs = torch.sum(
            micro_probs * attn_weights.unsqueeze(-1),
            dim=1
        )

        features = torch.cat([weighted_probs, uncertainty_features], dim=-1)
        logits = self.classifier(features)

        loss = None
        if labels is not None:
            loss = nn.CrossEntropyLoss()(logits, labels)

        return {
            "loss": loss,
            "logits": logits,
            "attention_weights": attn_weights
        }
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

macro_model = LearnedAttentionMacroAggregator(
    num_micro_labels=3,
    uncertainty_dim=uncertainty_dim,
    hidden_size=macro_hidden_size,
    dropout=macro_dropout,
    num_macro_labels=4
).to(device)

optimizer = torch.optim.AdamW(
    macro_model.parameters(),
    lr=macro_lr,
    weight_decay=macro_weight_decay
)

In [285]:
def train_macro_one_epoch(model, loader, optimizer, device):
    model.train()
    total_loss = 0.0

    for batch in loader:
        optimizer.zero_grad()

        micro_probs = batch["micro_probs"].to(device)
        uncertainty_features = batch["uncertainty_features"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(
            micro_probs=micro_probs,
            uncertainty_features=uncertainty_features,
            labels=labels
        )

        loss = outputs["loss"]
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)


def evaluate_macro_model(model, loader, device):
    model.eval()

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in loader:
            micro_probs = batch["micro_probs"].to(device)
            uncertainty_features = batch["uncertainty_features"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(
                micro_probs=micro_probs,
                uncertainty_features=uncertainty_features
            )

            preds = torch.argmax(outputs["logits"], dim=-1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    return np.mean(np.array(all_preds) == np.array(all_labels))

MACRO_EPOCHS = 20

for epoch in range(MACRO_EPOCHS):
    train_loss = train_macro_one_epoch(
        macro_model,
        train_macro_loader,
        optimizer,
        device
    )

    dev_acc = evaluate_macro_model(
        macro_model,
        dev_macro_loader,
        device
    )

    print(f"Epoch {epoch + 1}/{MACRO_EPOCHS}")
    print("Macro train loss:", train_loss)
    print("Macro dev accuracy:", dev_acc)

Epoch 1/20
Macro train loss: 0.9351745369759473
Macro dev accuracy: 0.6623376623376623
Epoch 2/20
Macro train loss: 0.7379348034208472
Macro dev accuracy: 0.6688311688311688
Epoch 3/20
Macro train loss: 0.6876636451327955
Macro dev accuracy: 0.6688311688311688
Epoch 4/20
Macro train loss: 0.6507141051160825
Macro dev accuracy: 0.7012987012987013
Epoch 5/20
Macro train loss: 0.6277681656859138
Macro dev accuracy: 0.6883116883116883
Epoch 6/20
Macro train loss: 0.60556909325835
Macro dev accuracy: 0.7077922077922078
Epoch 7/20
Macro train loss: 0.6278921119965516
Macro dev accuracy: 0.7207792207792207
Epoch 8/20
Macro train loss: 0.5996306798868365
Macro dev accuracy: 0.7207792207792207
Epoch 9/20
Macro train loss: 0.6114134466299763
Macro dev accuracy: 0.7077922077922078
Epoch 10/20
Macro train loss: 0.5977175547318025
Macro dev accuracy: 0.7142857142857143
Epoch 11/20
Macro train loss: 0.5977443117503222
Macro dev accuracy: 0.7142857142857143
Epoch 12/20
Macro train loss: 0.59154833185

# 3.Testing and Evaluation
(You can add as many code blocks and text blocks as you need. However, YOU SHOULD NOT MODIFY the section title)

## 3.1 End-to-end Claim Verification Inference
在完成 micro-level classifier 与 macro-level aggregator 的训练后，我们进一步对整个 Micro–Macro framework 进行 end-to-end inference。

对于每个 claim：

系统首先使用 micro-level BERT classifier 对所有 evidence passages 分别生成 sentence-level entailment probabilities
随后，macro-level attention aggregator 会综合多个 micro predictions
并输出最终的 claim-level prediction

最终系统能够生成：
claim
→ final claim label
→ supporting evidence ids

其中最终 claim labels 包括：SUPPORTS / REFUTES / NOT_ENOUGH_INFO/ DISPUTED

随后，我们将 prediction results 保存为 JSON 文件，并使用课程提供的 evaluation script 对 development set 进行整体评估。

这一过程能够验证整个 Micro–Macro pipeline 在 climate claim verification task 上的 end-to-end performance。

In [286]:
def predict_macro_with_learned_attention(model, loader, device):
    model.eval()
    predictions = {}

    with torch.no_grad():
        for batch in loader:
            micro_probs = batch["micro_probs"].to(device)
            uncertainty_features = batch["uncertainty_features"].to(device)

            outputs = model(
                micro_probs=micro_probs,
                uncertainty_features=uncertainty_features
            )

            preds = torch.argmax(outputs["logits"], dim=-1).cpu().numpy()

            for claim_id, evidence_ids, pred_id in zip(
                batch["claim_ids"],
                batch["evidence_ids"],
                preds
            ):
                predictions[claim_id] = {
                    "claim_label": macro_id2label[int(pred_id)],
                    "evidences": evidence_ids
                }

    return predictions

import json

dev_predictions = predict_macro_with_learned_attention(
    macro_model,
    dev_macro_loader,
    device
)

with open("dev_predictions_learned_attention_macro.json", "w", encoding="utf-8") as f:
    json.dump(dev_predictions, f, indent=2)

with open("dev_claims_small.json", "w", encoding="utf-8") as f:
    json.dump(dev_claims_small, f, indent=2)

!python eval.py \
    --predictions dev_predictions_learned_attention_macro.json \
    --groundtruth dev_claims_small.json

Evidence Retrieval F-score (F)    = 1.0
Claim Classification Accuracy (A) = 0.7337662337662337
Harmonic Mean of F and A          = 0.8464419475655431


## 3.2 Micro-level Classification Evaluation

为了进一步分析 micro-level classifier 的 sentence-level entailment prediction performance，我们对 development set 上的所有 claim-evidence pairs 进行 evaluation。

具体而言，系统会分别计算：

* precision
* recall
* F1-score

并生成：

* classification report
* confusion matrix

用于分析不同 micro labels 的分类效果。

其中：

* SUPPORTS
* REFUTES
* NOT_ENOUGH_INFO

分别对应 sentence-level evidence relations。

通过 confusion matrix，我们能够进一步观察模型容易混淆的类别。例如，部分 NOT_ENOUGH_INFO examples 可能会被错误分类为 SUPPORTS 或 REFUTES，反映出 evidence ambiguity 对 entailment prediction 带来的挑战。

In [287]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

def evaluate_micro_per_class(model, dataset, tokenizer=None):
    model.eval()

    all_preds = []
    all_labels = []

    for i in range(len(dataset)):
        item = dataset[i]

        inputs = {
            "input_ids": item["input_ids"].unsqueeze(0).to(model.device),
            "attention_mask": item["attention_mask"].unsqueeze(0).to(model.device),
        }

        if "token_type_ids" in item:
            inputs["token_type_ids"] = item["token_type_ids"].unsqueeze(0).to(model.device)

        with torch.no_grad():
            outputs = model(**inputs)
            pred = torch.argmax(outputs.logits, dim=-1).item()

        all_preds.append(pred)
        all_labels.append(item["labels"].item())

    print("Micro-level classification report:")
    print(
        classification_report(
            all_labels,
            all_preds,
            target_names=[
                "SUPPORTS",
                "REFUTES",
                "NOT_ENOUGH_INFO"
            ],
            digits=4
        )
    )

    print("Confusion matrix:")
    print(confusion_matrix(all_labels, all_preds))

    return all_labels, all_preds
micro_true, micro_pred = evaluate_micro_per_class(
    model,
    dev_micro_dataset
)

Micro-level classification report:
                 precision    recall  f1-score   support

       SUPPORTS     0.6373    0.7602    0.6933       171
        REFUTES     0.2941    0.3509    0.3200        57
NOT_ENOUGH_INFO     0.6211    0.4878    0.5464       205

       accuracy                         0.5774       433
      macro avg     0.5175    0.5330    0.5199       433
   weighted avg     0.5844    0.5774    0.5746       433

Confusion matrix:
[[130   6  35]
 [ 11  20  26]
 [ 63  42 100]]


## 3.3 Macro-level Classification Evaluation

除了 micro-level evaluation 外，我们还进一步对最终的 macro-level claim predictions 进行整体评估。

具体而言，系统会将：

* predicted macro labels
* ground-truth claim labels

进行比较，并计算：

* precision
* recall
* F1-score
* confusion matrix

其中 macro-level labels 包括：

* SUPPORTS
* REFUTES
* NOT_ENOUGH_INFO
* DISPUTED

相比 micro-level classification，macro-level evaluation 更关注多个 evidence predictions 的综合结果，因此 DISPUTED 类别的预测能力能够反映 learned attention aggregation 是否成功学习 evidence-level conflicts。

通过这一 evaluation，我们能够整体分析 Micro–Macro framework 在 climate claim verification task 上的最终性能表现。

In [288]:
from sklearn.metrics import classification_report, confusion_matrix

def evaluate_macro_predictions(predictions, groundtruth_claims):
    y_true = []
    y_pred = []

    for claim_id, pred_item in predictions.items():
        if claim_id not in groundtruth_claims:
            continue

        true_label = groundtruth_claims[claim_id]["claim_label"]
        pred_label = pred_item["claim_label"]

        y_true.append(true_label)
        y_pred.append(pred_label)

    labels = ["SUPPORTS", "REFUTES", "NOT_ENOUGH_INFO", "DISPUTED"]

    print("Macro-level classification report:")
    print(
        classification_report(
            y_true,
            y_pred,
            labels=labels,
            target_names=labels,
            digits=4
        )
    )

    print("Confusion matrix:")
    print(confusion_matrix(y_true, y_pred, labels=labels))

    return y_true, y_pred
macro_true, macro_pred = evaluate_macro_predictions(
    dev_predictions,
    dev_claims
)

Macro-level classification report:
                 precision    recall  f1-score   support

       SUPPORTS     0.7079    0.9265    0.8025        68
        REFUTES     0.6667    0.4444    0.5333        27
NOT_ENOUGH_INFO     0.8810    0.9024    0.8916        41
       DISPUTED     0.2000    0.0556    0.0870        18

       accuracy                         0.7338       154
      macro avg     0.6139    0.5822    0.5786       154
   weighted avg     0.6874    0.7338    0.6954       154

Confusion matrix:
[[63  2  2  1]
 [12 12  0  3]
 [ 4  0 37  0]
 [10  4  3  1]]


In [289]:
torch.save(macro_model.state_dict(), "macro_model.pt")

# Hugging Face 上传权重

In [290]:
torch.save({
    "model_state_dict": macro_model.state_dict(),

    "macro_hidden_size": macro_hidden_size,
    "macro_dropout": macro_dropout,
    "macro_lr": macro_lr,
    "macro_weight_decay": macro_weight_decay,

    "uncertainty_dim": uncertainty_dim,

    "num_micro_labels": 3,
    "num_macro_labels": 4,

    "macro_label2id": macro_label2id,
    "macro_id2label": macro_id2label

}, "macro_aggregator.pt")

In [291]:
import os

print(os.path.exists("macro_aggregator.pt"))

True


In [292]:
!pip install huggingface_hub -q

In [296]:
from huggingface_hub import login
login()

In [299]:
from huggingface_hub import HfApi

api = HfApi()

repo_id = "Ran9731/macro-aggregator"

api.create_repo(
    repo_id=repo_id,
    repo_type="model",
    private=True,
    exist_ok=True
)

RepoUrl('https://huggingface.co/Ran9731/macro-aggregator', endpoint='https://huggingface.co', repo_type='model', repo_id='Ran9731/macro-aggregator')

In [300]:
api.upload_file(
    path_or_fileobj="macro_aggregator.pt",
    path_in_repo="macro_aggregator.pt",
    repo_id=repo_id,
    repo_type="model"
)

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  macro_aggregator.pt         : 100%|##########| 7.88kB / 7.88kB            

CommitInfo(commit_url='https://huggingface.co/Ran9731/macro-aggregator/commit/65929dcdf7ba51385173c66b08a2ae8559a345d0', commit_message='Upload macro_aggregator.pt with huggingface_hub', commit_description='', oid='65929dcdf7ba51385173c66b08a2ae8559a345d0', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Ran9731/macro-aggregator', endpoint='https://huggingface.co', repo_type='model', repo_id='Ran9731/macro-aggregator'), pr_revision=None, pr_num=None)

In [301]:
macro_model_code = '''
import torch
import torch.nn as nn

class LearnedAttentionMacroAggregator(nn.Module):
    def __init__(
        self,
        num_micro_labels=3,
        uncertainty_dim=1,
        hidden_size=64,
        dropout=0.5,
        num_macro_labels=4
    ):
        super().__init__()

        self.attention = nn.Sequential(
            nn.Linear(num_micro_labels, hidden_size),
            nn.Tanh(),
            nn.Linear(hidden_size, 1)
        )

        self.classifier = nn.Sequential(
            nn.Linear(num_micro_labels + uncertainty_dim, hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, num_macro_labels)
        )

    def forward(
        self,
        micro_probs,
        uncertainty_features,
        evidence_mask=None,
        labels=None
    ):

        attn_logits = self.attention(micro_probs).squeeze(-1)

        if evidence_mask is not None:
            attn_logits = attn_logits.masked_fill(
                evidence_mask == 0,
                -1e9
            )

        attn_weights = torch.softmax(attn_logits, dim=1)

        weighted_probs = torch.sum(
            micro_probs * attn_weights.unsqueeze(-1),
            dim=1
        )

        features = torch.cat(
            [weighted_probs, uncertainty_features],
            dim=-1
        )

        logits = self.classifier(features)

        loss = None

        if labels is not None:
            loss = nn.CrossEntropyLoss()(logits, labels)

        return {
            "loss": loss,
            "logits": logits,
            "attention_weights": attn_weights
        }
'''

with open("macro_model.py", "w") as f:
    f.write(macro_model_code)

print("macro_model.py saved.")

macro_model.py saved.


In [302]:
api.upload_file(
    path_or_fileobj="macro_model.py",
    path_in_repo="macro_model.py",
    repo_id=repo_id,
    repo_type="model"
)

CommitInfo(commit_url='https://huggingface.co/Ran9731/macro-aggregator/commit/b1f958be41c4985e4c7b57a64cf3ed7e207b3c35', commit_message='Upload macro_model.py with huggingface_hub', commit_description='', oid='b1f958be41c4985e4c7b57a64cf3ed7e207b3c35', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Ran9731/macro-aggregator', endpoint='https://huggingface.co', repo_type='model', repo_id='Ran9731/macro-aggregator'), pr_revision=None, pr_num=None)

## Object Oriented Programming codes here

*You can use multiple code snippets. Just add more if needed*